In [ ]:
import numpy as np

def modularity_optimization(A, rng, gamma=1.0):
    """
    Optimize the modularity of a network by moving nodes between communities.

    A       : symmetric weighted adjacency matrix (n × n), zero diagonal
    rng     : numpy random number generator (for reproducibility)
    gamma   : resolution; higher -> more, smaller communities
    Returns : membership (community label for each node)
    """
    n = len(A)
    k = A.sum(axis=1)
    two_m = A.sum()
    c = np.arange(n)              # every node in its own community
    sigma_tot = k.copy()          # so each community's total is just that node's degree

    improved = True
    while improved:               # repeat sweeps until no node wants to move
        improved = False
        for i in rng.permutation(n):
            if move_node(i, A, k, c, sigma_tot, two_m, gamma):
                improved = True
    return c

def community_aggregation(A, c):
    """Aggregate communities into super-nodes.
    Returns the new adjacency matrix and the renumbered labels (node -> super-node)."""
    c = np.unique(c, return_inverse=True)[1]     # relabel to 0..nC-1
    nC = c.max() + 1
    A_new = np.zeros((nC, nC))
    for i in range(len(A)):
        for j in range(len(A)):
            A_new[c[i], c[j]] += A[i, j]
    return A_new, c

def move_node(i, A, k, c, sigma_tot, two_m, gamma=1.0):
    """Move node i to the neighbouring community with the largest modularity gain.
    Updates c and sigma_tot in place. Returns True if i changed community."""
    old = c[i]

    # 1. Take i out of its community
    sigma_tot[old] -= k[i]

    # 2. Weight from i to each neighbouring community: k_i,in(C)
    nb = np.nonzero(A[i])[0]
    nb = nb[nb != i]                                   # ignore self-loop
    k_i_in = np.bincount(c[nb], weights=A[i, nb], minlength=len(sigma_tot))

    # 3. Gain for each candidate community; default is going back to the old one
    best, best_gain = old, 0.0
    for C in np.unique(c[nb]):
        gain = k_i_in[C] - gamma * sigma_tot[C] * k[i] / two_m
        if gain > best_gain + 1e-12:                   # tolerance against float noise
            best, best_gain = C, gain

    # 4. Put i in the best community
    c[i] = best
    sigma_tot[best] += k[i]
    return best != old


def compute_network_modularity(A: np.ndarray, c: np.ndarray) -> float:
    two_m = A.sum()
    k = A.sum(axis=1)
    n = len(A)
    return sum((A[i, j] - k[i] * k[j] / two_m) * (c[i] == c[j])
               for i in range(n) for j in range(n)) / two_m


def louvain_algorithm(A, gamma=1.0, seed=None, verbose=True):
    """Louvain community detection from scratch.

    A       : symmetric weighted adjacency matrix (n × n), zero diagonal
    gamma   : resolution; higher -> more, smaller communities
    seed    : random seed for the node visiting order (for reproducibility)
    Returns : membership (community label for each original node), final modularity Q
    """
    rng = np.random.default_rng(seed)
    A = np.array(A, dtype=float)
    A_orig = A.copy()                        # Q is always measured on the original graph
    membership = np.arange(len(A))           # original node -> current super-node
    level = 0

    if verbose:
        print(f"level 0: {len(A)} nodes, Q = {compute_network_modularity(A_orig, membership):.4f}")

    while True:
        # Phase 1: move nodes (or super-nodes) between communities
        c = modularity_optimization(A, rng, gamma)

        # Stop when phase 1 merged nothing: every node is still on its own
        if len(np.unique(c)) == len(A):
            break

        # Phase 2: turn communities into super-nodes
        A, c = community_aggregation(A, c)
        membership = c[membership]
        level += 1

        if verbose:
            print(f"level {level}: {len(A)} communities, "
                  f"Q = {compute_network_modularity(A_orig, membership):.4f}")

    return membership, compute_network_modularity(A_orig, membership)

    

   


In [12]:
import networkx as nx

G = nx.karate_club_graph()
A = nx.to_numpy_array(G)          # 34 × 34 weighted adjacency matrix (weight=None for unweighted)
n = len(A)

print(f"nodes: {n}, edges: {int((A > 0).sum() / 2)}, total weight m: {A.sum() / 2:.0f}")

# Before: every node in its own community
c0 = np.arange(n)
print(c0)
print(f"before: {len(np.unique(c0))} communities, Q = {compute_network_modularity(A, c0):.4f}")

# After phase 1
rng = np.random.default_rng(7)
c1 = modularity_optimization(A, rng)
c1 = np.unique(c1, return_inverse=True)[1]          # renumber labels to 0, 1, 2, ...
print(f"after:  {len(np.unique(c1))} communities, Q = {compute_network_modularity(A, c1):.4f}, "
      f"sizes = {np.bincount(c1).tolist()}")

# Checks against networkx
comms = [set(np.where(c1 == C)[0]) for C in np.unique(c1)]
print(f"networkx Q of our partition: {nx.community.modularity(G, comms):.4f}")   # must match ours
nxl = nx.community.louvain_communities(G, seed=6)
print(f"networkx full Louvain: {len(nxl)} communities, Q = {nx.community.modularity(G, nxl):.4f}")

# The real-world split
club = np.array([G.nodes[v]["club"] == "Officer" for v in G]).astype(int)
print(f"real split: Q = {compute_network_modularity(A, club):.4f}")

nodes: 34, edges: 78, total weight m: 231
[ 0  1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18 19 20 21 22 23
 24 25 26 27 28 29 30 31 32 33]
before: 34 communities, Q = -0.0511
after:  7 communities, Q = 0.3663, sizes = [11, 2, 2, 3, 6, 4, 6]
networkx Q of our partition: 0.3663
networkx full Louvain: 4 communities, Q = 0.4439
real split: Q = 0.3914


## Danish Parliament: communities of MPs

Load the MP × MP agreement-rate matrix built in `similarity_matrix.ipynb` (absences excluded) and run our Louvain on it.

In [ ]:
import pandas as pd

d = np.load("similarity.npz")
A_ft, actors, party = d["A"], d["actor"], d["party"]
print(f"{len(A_ft)} MPs, mean agreement rate {A_ft[A_ft > 0].mean():.3f}")

membership, Q = louvain_algorithm(A_ft, seed=0)
print(f"\n{len(np.unique(membership))} communities, Q = {Q:.4f}")

Which parties end up in each community (rows = parties, columns = communities):

In [ ]:
pd.crosstab(pd.Series(party, name="party"), pd.Series(membership, name="community"))

Check against networkx on the same weighted graph:

In [ ]:
G_ft = nx.from_numpy_array(A_ft)
comms = [set(np.where(membership == C)[0]) for C in np.unique(membership)]
print(f"networkx Q of our partition: {nx.community.modularity(G_ft, comms):.4f}")
nxl = nx.community.louvain_communities(G_ft, weight="weight", seed=0)
print(f"networkx full Louvain: {len(nxl)} communities, Q = {nx.community.modularity(G_ft, nxl):.4f}")

## Graph before and after Louvain

- **Before**: the input graph, MPs coloured by their **party**. Louvain never sees these labels; they show the grouping we start from.
- **After**: the same graph coloured by the **community** Louvain found. Each community takes the colour of its largest party, so you can follow parties merging into blocs. Edges inside a community take its colour; edges between communities stay grey.

**Both panels use the same positions.** Louvain only assigns labels and never moves nodes. A random layout before and a tidy one after would credit the algorithm with work the layout algorithm did.

Drawing choices (they do not affect the algorithm, which runs on the full `A_ft`):
- **Edges**: each MP's `K` strongest agreement ties; all ~16,000 weighted edges would draw as a solid block.
- **Layout**: force-directed (`spring_layout`) on the agreement *above the average* (negative values set to 0). On raw `A` the rates are all high (mean ~0.8), so every MP pulls equally on every other one and the layout is a single blob.
- Colours are the dataviz reference palette in fixed order (8 hues); parties 9+ reuse a hue with a square marker, and independents are grey crosses.

The table below the plot shows how close-knit the communities are: mean agreement within each community (diagonal) against between communities.

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

n = len(A_ft)
INK, INK_2, EDGE, GREY = "#0b0b0b", "#52514e", "#d6d6d1", "#8a8a85"
PALETTE = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#6250d6", "#e34948"]
ABBR = {
    "Socialdemokratiet": "S", "Venstre, Danmarks Liberale Parti": "V", "Moderaterne": "M",
    "Socialistisk Folkeparti": "SF", "Danmarksdemokraterne – Inger Støjberg": "DD",
    "Liberal Alliance": "LA", "Det Konservative Folkeparti": "KF", "Enhedslisten": "EL",
    "Alternativet": "ALT", "Dansk Folkeparti": "DF", "Radikale Venstre": "RV",
}
mp_party = np.array(["Ind." if p.startswith("Uden for") else ABBR.get(p, p) for p in party])

# Party style: fixed order by party size; 8 hues, parties 9+ reuse a hue with a square marker
parties = [p for p in pd.Series(mp_party).value_counts().index if p != "Ind."]
style = {p: (PALETTE[k % 8], "o" if k < 8 else "s") for k, p in enumerate(parties)}
style["Ind."] = (GREY, "X")

# Each community takes the colour of its largest party, so the eye can follow parties into blocs
comm_main = {c: pd.Series(mp_party[membership == c]).value_counts().index[0] for c in np.unique(membership)}
comm_color = {c: style[p][0] for c, p in comm_main.items()}

# Same layout in both panels (Louvain assigns labels, it never moves nodes).
# Force-directed on the agreement *above average*: on raw A (mean ~0.8) every MP pulls equally on all others.
W = np.clip(A_ft - A_ft[A_ft > 0].mean(), 0, None)
pos = nx.spring_layout(nx.from_numpy_array(W), weight="weight", seed=0)
xy = np.array([pos[i] for i in range(n)])

# Edges drawn: each MP's K strongest ties (all ~16,000 would be a solid block)
K = 5
rows, cols = np.repeat(np.arange(n), K), np.argsort(-A_ft, axis=1)[:, :K].ravel()
G_draw = nx.Graph()
G_draw.add_nodes_from(range(n))
G_draw.add_edges_from(zip(rows, cols))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 7.5))
node_kw = dict(node_size=55, edgecolors="white", linewidths=0.8)   # white ring separates overlapping nodes

# --- Before: input graph, coloured by party (Louvain never sees these labels)
nx.draw_networkx_edges(G_draw, pos, ax=ax1, edge_color=EDGE, width=0.6)
for p, (col, mk) in style.items():
    nodes = np.flatnonzero(mp_party == p).tolist()
    nx.draw_networkx_nodes(G_draw, pos, nodelist=nodes, node_color=col, node_shape=mk, ax=ax1, **node_kw)
ax1.set_title(f"Before: {n} MPs coloured by party", color=INK, loc="left")
ax1.legend(handles=[Line2D([], [], ls="", marker=mk, color=col, markeredgecolor="white", markersize=8,
                           label=f"{p} ({(mp_party == p).sum()})") for p, (col, mk) in style.items()],
           ncol=6, loc="upper center", bbox_to_anchor=(0.5, 0), frameon=False, fontsize=9, labelcolor=INK_2)

# --- After: same graph, coloured by Louvain community
inside = [(u, v) for u, v in G_draw.edges if membership[u] == membership[v]]
between = [(u, v) for u, v in G_draw.edges if membership[u] != membership[v]]
nx.draw_networkx_edges(G_draw, pos, ax=ax2, edgelist=between, edge_color=EDGE, width=0.6)
nx.draw_networkx_edges(G_draw, pos, ax=ax2, edgelist=inside, width=0.6, alpha=0.35,
                       edge_color=[comm_color[membership[u]] for u, _ in inside])
nx.draw_networkx_nodes(G_draw, pos, ax=ax2, node_color=[comm_color[c] for c in membership], **node_kw)
ax2.set_title(f"After: {len(comm_main)} Louvain communities (Q = {Q:.3f})", color=INK, loc="left")

# Legend: community, its main parties (>= 10% of members), size and how close-knit it is
handles = []
for c in comm_main:
    sel = membership == c
    counts = pd.Series(mp_party[sel]).value_counts()
    main = ", ".join(counts.index[counts >= 0.1 * sel.sum()])
    inner = A_ft[np.ix_(sel, sel)][np.triu_indices(sel.sum(), 1)]
    handles.append(Line2D([], [], ls="", marker="o", color=comm_color[c], markeredgecolor="white", markersize=8,
                          label=f"Community {c}: {main} · {sel.sum()} MPs · mean agreement {inner[inner > 0].mean():.2f}"))
ax2.legend(handles=handles, loc="upper center", bbox_to_anchor=(0.5, 0), frameon=False,
           fontsize=9, labelcolor=INK_2)

for ax in (ax1, ax2):
    ax.axis("off")
fig.text(0.5, -0.06, f"Same positions in both panels; edges = each MP's {K} strongest agreement ties. "
         "Agreement = share of shared sessions voted the same way.", ha="center", fontsize=8.5, color=INK_2)
plt.tight_layout()
plt.show()

# Between-community agreement, for comparison with the within-community values above
cs = list(comm_main)
def mean_agreement(a, b):
    block = A_ft[np.ix_(membership == a, membership == b)]
    return block[block > 0].mean()

print("Mean agreement within (diagonal) and between communities:")
pd.DataFrame([[mean_agreement(a, b) for b in cs] for a in cs],
             index=[f"C{c}" for c in cs], columns=[f"C{c}" for c in cs]).round(3)


## The chamber before and after Louvain

The same before/after comparison, drawn as a parliament diagram.

**This is a snapshot of one day, not all 190 MPs.** Our 190 MPs include substitutes who replaced each other over three years, so seating all of them would show a chamber that never existed. Instead we seat the **179 MPs who held a seat at the last session in the data**, using the raw votes before any filtering, with their party on that day.
- Some of them are **not in the analysis** (fewer than 100 valid votes, or no edges). They are drawn as hollow grey seats instead of being hidden.
- Communities come from voting over the **whole** 2022–25 period. An MP who switched party is shown in their new party but keeps the community their full voting record gave them.
- The seating is **stylised**: parties are ordered left to right on the political spectrum, not by the real seat plan. Within a party, MPs are grouped by community, so an MP who left their party's bloc shows up at the edge of the party's wedge.
- Small parties and independents (e.g. Borgernes Parti, the Faroese seat) are folded into the grey *Ind./other* group, to keep within the 8-hue palette.

In [ ]:
import json

# --- Who sat in the chamber: the 179 seats at the last session in the data (raw votes, before any filtering)
votes_all = pd.read_csv("votes.csv", parse_dates=["date"])
last_session = votes_all["session"].max()
snap_date = votes_all.loc[votes_all["session"] == last_session, "date"].iloc[0]
seated = sorted({v["aktørid"] for v in json.load(open("raw/Stemme.json")) if v["afstemningid"] == last_session})

# Party of each seated MP on that date (same rule as retrieve.ipynb: most recent membership covering the date)
rel = pd.DataFrame(json.load(open("raw/AktørAktør.json")))
rel = pd.DataFrame({"actor": rel["fraaktørid"], "party": rel["TilAktør"].str["navn"],
                    "start": pd.to_datetime(rel["startdato"]),
                    "end": pd.to_datetime(rel["slutdato"]).fillna(pd.Timestamp.max)})
rel = rel[rel["actor"].isin(seated) & (rel["start"] <= snap_date) & (rel["end"] >= snap_date)]
seat_party_full = rel.sort_values("start").groupby("actor")["party"].last().reindex(seated)

def short(p):
    if pd.isna(p) or p.startswith("Uden for"):
        return "Ind."
    return ABBR.get(p, p)

seats = pd.DataFrame({"actor": seated, "party": [short(p) for p in seat_party_full]})
seats.loc[~seats["party"].isin(style), "party"] = "Ind."      # small parties (e.g. BP, Faroe/Greenland seats) fold into grey
comm_of = dict(zip(actors, membership))
seats["community"] = seats["actor"].map(comm_of)              # NaN = not in the analysis

# --- Seat order: parties left -> right on the political spectrum; inside a party, MPs grouped by community
SPECTRUM = ["EL", "ALT", "SF", "RV", "S", "M", "V", "KF", "LA", "DD", "DF", "Ind."]
comm_pos = {c: np.mean([SPECTRUM.index(p) for p in mp_party[membership == c] if p in SPECTRUM]) for c in comm_main}
seats["p_rank"] = seats["party"].map(SPECTRUM.index)
seats["c_rank"] = seats["community"].map(comm_pos).fillna(99)
seats = seats.sort_values(["p_rank", "c_rank", "actor"]).reset_index(drop=True)

def hemicycle(n, rows=9, r_in=1.0, r_out=2.4):
    """Seat coordinates of a half-circle chamber, ordered left -> right."""
    radii = np.linspace(r_in, r_out, rows)
    per_row = np.floor(n * radii / radii.sum()).astype(int)
    per_row[np.argsort(-radii)[: n - per_row.sum()]] += 1     # hand out the remainder to the outer rows
    pts = [(np.pi * (1 - (j + 0.5) / m), r) for r, m in zip(radii, per_row) for j in range(m)]
    pts.sort(key=lambda t: (-t[0], t[1]))                     # by angle (left first), then inner to outer
    return np.array([[r * np.cos(a), r * np.sin(a)] for a, r in pts])

seat_xy = hemicycle(len(seats))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 4.6))
seat_kw = dict(s=70, edgecolor="white", linewidth=0.8)

# Before: party
for p, (col, mk) in style.items():
    sel = (seats["party"] == p).to_numpy()
    if sel.any():
        ax1.scatter(*seat_xy[sel].T, color=col, marker=mk, label=f"{'Ind./other' if p == 'Ind.' else p} ({sel.sum()})", **seat_kw)
ax1.set_title(f"Before: {len(seats)} seats on {snap_date:%d %b %Y}, by party", color=INK, loc="left")
ax1.legend(ncol=6, loc="upper center", bbox_to_anchor=(0.5, 0), frameon=False, fontsize=9, labelcolor=INK_2)

# After: Louvain community (same seats)
for c in comm_main:
    sel = (seats["community"] == c).to_numpy()
    ax2.scatter(*seat_xy[sel].T, color=comm_color[c], label=f"Community {c} ({sel.sum()})", **seat_kw)
out = seats["community"].isna().to_numpy()
ax2.scatter(*seat_xy[out].T, s=70, facecolor="none", edgecolor=GREY, linewidth=1,
            label=f"Not in analysis ({out.sum()})")
ax2.set_title("After: the same seats, by Louvain community", color=INK, loc="left")
ax2.legend(ncol=4, loc="upper center", bbox_to_anchor=(0.5, 0), frameon=False, fontsize=9, labelcolor=INK_2)

for ax in (ax1, ax2):
    ax.set_aspect("equal")
    ax.axis("off")
fig.text(0.5, -0.04, "Stylised seating: parties ordered left to right on the political spectrum, not the actual seat plan. "
         "Communities come from votes over the whole 2022–25 period. "
         "Not in analysis = fewer than 100 valid votes or no edges.", ha="center", fontsize=8.5, color=INK_2)
plt.tight_layout()
plt.show()

